# 18 — Final untouched-test evaluation for V1

This notebook performs the single final evaluation of the frozen baseline LightGBM on the previously untouched test set. It creates evaluation reports and metadata only. The model, preprocessing, threshold, labels, features, and split are never modified after test results become visible.

### 1. Locate and fingerprint the frozen artifacts

**What this cell does:** Resolves exact current-experiment paths, creates output directories, and fingerprints the frozen model, preprocessor, test artifacts, and metadata.  
**Why it matters:** The final assessment is valid only if it uses the 153k-era frozen artifacts rather than historical 82k files.  
**What to understand:** Hashes recorded here must remain identical after evaluation; the fixed threshold is 0.50 and is not optimized below.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, balanced_accuracy_score, confusion_matrix, precision_recall_curve, roc_curve
)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns",80); pd.set_option("display.width",180)
PROJECT_ROOT=Path.cwd().resolve()
if not (PROJECT_ROOT/"data/modeling/preprocessed/X_test_tree.csv").exists() and (PROJECT_ROOT.parent/"data/modeling/preprocessed/X_test_tree.csv").exists(): PROJECT_ROOT=PROJECT_ROOT.parent
assert (PROJECT_ROOT/"data/interim/feature_dataset.csv").exists(), "Could not locate AdoptAI_V1 root."

PATHS={
 "model":PROJECT_ROOT/"models/baseline/lightgbm.joblib",
 "preprocessor":PROJECT_ROOT/"models/preprocessing/tree_preprocessor.joblib",
 "x_test":PROJECT_ROOT/"data/modeling/preprocessed/X_test_tree.csv",
 "y_test":PROJECT_ROOT/"data/modeling/preprocessed/y_test.csv",
 "test_identifiers":PROJECT_ROOT/"data/modeling/preprocessed/test_identifiers.csv",
 "train":PROJECT_ROOT/"data/modeling/train.csv", "validation":PROJECT_ROOT/"data/modeling/validation.csv", "test_raw":PROJECT_ROOT/"data/modeling/test.csv",
 "split_summary":PROJECT_ROOT/"reports/final_split_summary.csv", "split_by_run":PROJECT_ROOT/"reports/final_split_by_run.csv",
 "feature_metadata":PROJECT_ROOT/"reports/preprocessing_feature_report.csv",
 "baseline_metrics":PROJECT_ROOT/"reports/baseline_model_comparison.csv",
 "validation_redesign":PROJECT_ROOT/"reports/validation_split_recommendation.csv",
 "shift_summary":PROJECT_ROOT/"reports/distribution_shift_summary.csv",
 "feature_dataset":PROJECT_ROOT/"data/interim/feature_dataset.csv",
 "cleaned_metrics":PROJECT_ROOT/"data/processed/cleaned_metrics.csv",
}
REPORT_DIR=PROJECT_ROOT/"reports"; FIGURE_DIR=REPORT_DIR/"figures/final_test"; MODEL_DIR=PROJECT_ROOT/"models/final_v1"
FIGURE_DIR.mkdir(parents=True,exist_ok=True); MODEL_DIR.mkdir(parents=True,exist_ok=True)
THRESHOLD=0.50
def sha256(path,block=1024*1024):
 d=hashlib.sha256()
 with open(path,"rb") as h:
  for chunk in iter(lambda:h.read(block),b""): d.update(chunk)
 return d.hexdigest()
missing=[str(p) for p in PATHS.values() if not p.exists()]; assert not missing,missing
frozen_names=["model","preprocessor","x_test","y_test","test_identifiers","test_raw","split_summary","split_by_run","feature_metadata","feature_dataset","cleaned_metrics"]
frozen_before={name:sha256(PATHS[name]) for name in frozen_names}
print(f"Repository: {PROJECT_ROOT}")
print("Frozen baseline model:",PATHS["model"])
print("Frozen tree preprocessor:",PATHS["preprocessor"])
print("Decision threshold: 0.50 — fixed development/demo reference, not a production-optimized threshold.")

Repository: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1
Frozen baseline model: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/baseline/lightgbm.joblib
Frozen tree preprocessor: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/preprocessing/tree_preprocessor.joblib
Decision threshold: 0.50 — fixed development/demo reference, not a production-optimized threshold.


### 2. Validate experiment identity, test integrity, and run separation

**What this cell does:** Loads the frozen matrix, target, identifiers, split reports, feature metadata, and baseline model, then checks dimensions, values, class balance, machines, runs, and disjoint run IDs.  
**Why it matters:** Misaligned rows or overlapping runs would invalidate final metrics.  
**What to understand:** The expected current test is exactly 27,944 rows; transformed features must be finite and the model must expect the same 360 columns.

In [2]:
X_test=pd.read_csv(PATHS["x_test"]); y_test=pd.read_csv(PATHS["y_test"]).iloc[:,0].astype(int); identifiers=pd.read_csv(PATHS["test_identifiers"])
split_summary=pd.read_csv(PATHS["split_summary"]); split_runs=pd.read_csv(PATHS["split_by_run"]); feature_meta=pd.read_csv(PATHS["feature_metadata"])
baseline_report=pd.read_csv(PATHS["baseline_metrics"]); redesign=pd.read_csv(PATHS["validation_redesign"]); shift_summary=pd.read_csv(PATHS["shift_summary"])
model=joblib.load(PATHS["model"])
assert len(X_test)==len(y_test)==len(identifiers)==27_944
assert X_test.shape[1]==getattr(model,"n_features_in_",X_test.shape[1])==360
numeric=X_test.to_numpy(dtype=float); assert not np.isnan(numeric).any(); assert np.isfinite(numeric).all()
assert set(y_test.unique()).issubset({0,1}) and y_test.nunique()==2
assert identifiers.machine_id.nunique()==4 and identifiers.run_id.nunique()==9
sets={s:set(split_runs.loc[split_runs.split.eq(s),"run_id"].astype(str)) for s in ["train","validation","test"]}
assert sets["train"].isdisjoint(sets["validation"]|sets["test"]); assert sets["validation"].isdisjoint(sets["test"])
test_row=split_summary.loc[split_summary.split.eq("test")].iloc[0]
assert int(test_row.total_rows)==len(X_test) and int(test_row.positive_rows)==int(y_test.sum())
retained_count=int(feature_meta.retained_or_removed.eq("retained").sum())
candidate_count=243; valid_modeling_rows=int(split_summary.total_rows.sum())
cleaned_rows=sum(1 for _ in open(PATHS["cleaned_metrics"],encoding="utf-8"))-1
assert cleaned_rows==153_521 and valid_modeling_rows==133_016 and retained_count==228
artifact_validation=pd.DataFrame([{
 "test_rows":len(X_test),"test_positive_rows":int(y_test.sum()),"test_negative_rows":int((1-y_test).sum()),
 "test_positive_rate":float(y_test.mean()),"test_machines":identifiers.machine_id.nunique(),"test_runs":identifiers.run_id.nunique(),
 "transformed_features":X_test.shape[1],"missing_transformed_values":int(X_test.isna().sum().sum()),
 "infinite_transformed_values":int(np.isinf(numeric).sum()),"run_overlap_count":sum(len(sets[a]&sets[b]) for a,b in [("train","validation"),("train","test"),("validation","test")])
}])
display(artifact_validation)
print("Frozen LightGBM parameters:"); display(pd.Series(model.get_params(),name="value").to_frame())

,test_rows,test_positive_rows,test_negative_rows,test_positive_rate,test_machines,test_runs,transformed_features,missing_transformed_values,infinite_transformed_values,run_overlap_count
0,27944,11356,16588,0.406384,4,9,360,0,0,0


Frozen LightGBM parameters:


,value
boosting_type,gbdt
class_weight,None
colsample_bytree,1.0
importance_type,split
learning_rate,0.05
max_depth,-1
min_child_samples,20
min_child_weight,0.001
min_split_gain,0.0
n_estimators,300


### 3. Record the test-protection history before inference

**What this cell does:** Converts existing split-redesign and distribution-shift guardrails into an explicit protection record and confirms no test run was used in development partitions.  
**Why it matters:** An unbiased final test requires that tuning, threshold selection, calibration, and model selection happened without test feedback.  
**What to understand:** These are provenance checks based on existing reports; this cell does not inspect model behavior on test.

In [3]:
redesign_row=redesign.iloc[0]
shift_guard=shift_summary.loc[shift_summary.metric.eq("test_model_performance_used"),"value"].astype(str).str.lower().iloc[0]
protection_history=pd.DataFrame([
 {"protection_check":"test_not_used_for_tuning","confirmed":True,"evidence":"Development notebooks and reports compare train/validation only; frozen baseline selected before final test."},
 {"protection_check":"test_not_used_for_threshold_optimization","confirmed":True,"evidence":"Threshold 0.50 is the unchanged reference; historical threshold work was not adopted for current V1."},
 {"protection_check":"test_not_used_for_calibration","confirmed":True,"evidence":"No calibration artifact exists or is loaded; notebook 17 stopped before calibration."},
 {"protection_check":"test_not_used_for_model_selection","confirmed":str(redesign_row.get("test_artifacts_loaded",False)).lower()=="false" and shift_guard=="false","evidence":"validation_split_recommendation test_artifacts_loaded=False and distribution_shift_summary test_model_performance_used=False."},
 {"protection_check":"test_runs_disjoint_from_development","confirmed":sets["test"].isdisjoint(sets["train"]|sets["validation"]),"evidence":"Complete run IDs in final_split_by_run.csv are pairwise disjoint."},
 {"protection_check":"fixed_test_run_count","confirmed":len(sets["test"])==9,"evidence":"Nine fixed latest test runs remain assigned to test."},
])
assert protection_history.confirmed.all()
display(protection_history)

,protection_check,confirmed,evidence
0,test_not_used_for_tuning,True,Development notebooks and reports compare trai...
1,test_not_used_for_threshold_optimization,True,Threshold 0.50 is the unchanged reference; his...
2,test_not_used_for_calibration,True,No calibration artifact exists or is loaded; n...
3,test_not_used_for_model_selection,True,validation_split_recommendation test_artifacts...
4,test_runs_disjoint_from_development,True,Complete run IDs in final_split_by_run.csv are...
5,fixed_test_run_count,True,Nine fixed latest test runs remain assigned to...


### 4. Generate the one-time frozen-model test scores

**What this cell does:** Calls `predict_proba` once on the frozen preprocessed test matrix, validates its range, converts scores to the fixed 0.50 class decision, and saves row-aligned predictions.  
**Why it matters:** These scores are the sole basis of final V1 test evaluation; neither model nor preprocessing is refitted.  
**What to understand:** `risk_score` is a 0–100 model score, not a calibrated probability or guaranteed real-world risk percentage.

In [4]:
raw_model_score=model.predict_proba(X_test)[:,1]
assert len(raw_model_score)==len(y_test) and np.isfinite(raw_model_score).all() and ((raw_model_score>=0)&(raw_model_score<=1)).all()
predicted_class=(raw_model_score>=THRESHOLD).astype("int8")
predictions=identifiers.copy()
predictions["true_label"]=y_test.to_numpy(); predictions["raw_model_score"]=raw_model_score
predictions["risk_score"]=100*raw_model_score; predictions["predicted_class_at_0_50"]=predicted_class
predictions["risk_score_description"]="Model risk score for slowdown in the next 5 minutes"
predictions.to_csv(REPORT_DIR/"final_test_predictions.csv",index=False)
print(f"Saved {len(predictions):,} predictions. Example: score {raw_model_score[0]:.4f} -> Risk Score {100*raw_model_score[0]:.1f}/100.")
print("Alert/reference boundary: Risk Score 50. Prototype/demo rule; not a validated production threshold.")

Saved 27,944 predictions. Example: score 0.8667 -> Risk Score 86.7/100.
Alert/reference boundary: Risk Score 50. Prototype/demo rule; not a validated production threshold.


### 5. Calculate the frozen global test metrics

**What this cell does:** Computes ranking and classification metrics at the unchanged 0.50 threshold, including confusion-matrix counts and specificity.  
**Why it matters:** PR-AUC is the main ranking metric, while FP and FN show operational alert tradeoffs.  
**What to understand:** FN means a real next-five-minute slowdown was missed; FP means an alert was generated when no slowdown occurred.

In [5]:
tn,fp,fn,tp=confusion_matrix(y_test,predicted_class,labels=[0,1]).ravel()
global_metrics={
 "evaluation":"final_untouched_test","threshold":THRESHOLD,"rows":len(y_test),"positive_rows":int(y_test.sum()),"negative_rows":int((1-y_test).sum()),
 "positive_rate":float(y_test.mean()),"pr_auc":average_precision_score(y_test,raw_model_score),"roc_auc":roc_auc_score(y_test,raw_model_score),
 "precision":precision_score(y_test,predicted_class,zero_division=0),"recall":recall_score(y_test,predicted_class,zero_division=0),
 "f1":f1_score(y_test,predicted_class,zero_division=0),"accuracy":accuracy_score(y_test,predicted_class),
 "balanced_accuracy":balanced_accuracy_score(y_test,predicted_class),"specificity":tn/(tn+fp),
 "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),"predicted_positive_rate":float(predicted_class.mean()),
 "threshold_status":"fixed_prototype_development_reference_not_production_optimized"
}
final_test_metrics=pd.DataFrame([global_metrics]); final_test_metrics.to_csv(REPORT_DIR/"final_test_metrics.csv",index=False)
display(final_test_metrics.T.rename(columns={0:"value"}))

,value
evaluation,final_untouched_test
threshold,0.5
rows,27944
positive_rows,11356
negative_rows,16588
positive_rate,0.406384
pr_auc,0.951936
roc_auc,0.954877
precision,0.815435
recall,0.922068


### 6. Evaluate machine and complete-run robustness

**What this cell does:** Computes the same diagnostic metrics for every test machine and run, with undefined PR/ROC values left missing for single-class subsets.  
**Why it matters:** Global performance can hide weak operating regimes, which prior analysis identified as the main V1 limitation.  
**What to understand:** Informative-run summaries use mixed-class runs with at least 500 rows; weak subsets are documented and never removed.

In [6]:
eval_frame=predictions.copy()
def group_metrics(frame,keys):
 rows=[]
 group_arg=keys[0] if len(keys)==1 else keys
 for key,g in frame.groupby(group_arg,sort=False):
  key=(key,) if len(keys)==1 else key; y=g.true_label.astype(int).to_numpy(); p=g.raw_model_score.to_numpy(); pred=g.predicted_class_at_0_50.astype(int).to_numpy()
  tnn,fpp,fnn,tpp=confusion_matrix(y,pred,labels=[0,1]).ravel(); mixed=np.unique(y).size==2
  row=dict(zip(keys,key)); row.update({"rows":len(g),"runs":g.run_id.nunique(),"positive_count":int(y.sum()),"negative_count":int((1-y).sum()),"positive_rate":float(y.mean()),
   "class_composition":"mixed" if mixed else ("all_positive" if y[0]==1 else "all_negative"),"pr_auc":average_precision_score(y,p) if mixed else np.nan,
   "roc_auc":roc_auc_score(y,p) if mixed else np.nan,"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),
   "f1":f1_score(y,pred,zero_division=0),"tn":int(tnn),"fp":int(fpp),"fn":int(fnn),"tp":int(tpp),"mean_predicted_probability":float(p.mean()),
   "predicted_positive_rate":float(pred.mean()),"informative_mixed_run":bool(mixed and len(g)>=500)})
  rows.append(row)
 return pd.DataFrame(rows)
by_machine=group_metrics(eval_frame,["machine_id"]); by_run=group_metrics(eval_frame,["machine_id","run_id"])
by_machine.to_csv(REPORT_DIR/"final_test_by_machine.csv",index=False); by_run.to_csv(REPORT_DIR/"final_test_by_run.csv",index=False)
informative=by_run[by_run.informative_mixed_run].dropna(subset=["pr_auc"])
strongest=informative.loc[informative.pr_auc.idxmax()]; weakest=informative.loc[informative.pr_auc.idxmin()]
run_stability={"informative_runs":len(informative),"strongest_run_id":strongest.run_id,"strongest_run_pr_auc":strongest.pr_auc,
 "weakest_informative_run_id":weakest.run_id,"weakest_informative_run_machine":weakest.machine_id,"minimum_informative_run_pr_auc":weakest.pr_auc,
 "minimum_informative_run_recall":informative.recall.min(),"mean_informative_run_pr_auc":informative.pr_auc.mean(),"std_informative_run_pr_auc":informative.pr_auc.std(ddof=0)}
print("By machine:"); display(by_machine)
print("By run:"); display(by_run.sort_values("pr_auc",na_position="last"))
print("Run stability:"); display(pd.Series(run_stability,name="value").to_frame())

By machine:


,machine_id,rows,runs,positive_count,negative_count,positive_rate,class_composition,pr_auc,roc_auc,precision,recall,f1,tn,fp,fn,tp,mean_predicted_probability,predicted_positive_rate,informative_mixed_run
0,0890dcc046c079acc4de4202,8359,3,319,8040,0.038162,mixed,0.048376,0.571954,0.000000,0.000000,0.000000,8007,33,319,0,0.016214,0.003948,True
1,7232bc533c21ce408d45d473,4206,3,1400,2806,0.332858,mixed,0.723283,0.747359,0.483113,0.705000,0.573337,1750,1056,413,987,0.509148,0.485735,True
2,a0f8c86097e55fbfa506d057,12986,2,7302,5684,0.562298,mixed,0.968500,0.962224,0.848244,0.979047,0.908964,4405,1279,153,7149,0.674016,0.649007,True
3,d588df123ac0d0ce20b112ac,2393,1,2335,58,0.975763,mixed,1.000000,1.000000,0.999144,1.000000,0.999572,56,2,0,2335,0.976927,0.976598,True


By run:


,machine_id,run_id,rows,runs,positive_count,negative_count,positive_rate,class_composition,pr_auc,roc_auc,precision,recall,f1,tn,fp,fn,tp,mean_predicted_probability,predicted_positive_rate,informative_mixed_run
0,0890dcc046c079acc4de4202,795ee584-86ed-4fe5-b317-3d022b850274,5047,1,166,4881,0.032891,mixed,0.056990,0.680935,0.000000,0.000000,0.000000,4848,33,166,0,0.024263,0.006539,True
2,0890dcc046c079acc4de4202,6ae194e7-276b-4eef-94d3-d67c6be4b535,2898,1,153,2745,0.052795,mixed,0.064795,0.632380,0.000000,0.000000,0.000000,2745,0,153,0,0.003360,0.000000,True
5,7232bc533c21ce408d45d473,adc86181-d9a4-4cf6-a8be-4be144db7799,1685,1,389,1296,0.230861,mixed,0.732740,0.863263,0.455820,0.835476,0.589837,908,388,64,325,0.482038,0.423145,True
3,7232bc533c21ce408d45d473,5bca2646-6745-4bbb-987d-ccb0a4486506,2405,1,895,1510,0.372141,mixed,0.749350,0.743872,0.474843,0.674860,0.557453,842,668,291,604,0.525543,0.528898,True
6,a0f8c86097e55fbfa506d057,cf6e2a7e-5b3d-4f7b-a21f-e9ffd68b5268,5973,1,1071,4902,0.179307,mixed,0.852479,0.952829,0.649011,0.858077,0.739043,4405,497,152,919,0.297194,0.237067,True
7,a0f8c86097e55fbfa506d057,ee6b8dde-602e-4c14-b533-43bcfcba11ea,7013,1,6231,782,0.888493,mixed,0.977996,0.844819,0.888477,0.999840,0.940874,0,782,1,6230,0.994956,0.999857,True
8,d588df123ac0d0ce20b112ac,03b262cc-46f0-4837-a184-1eea22fdfdd0,2393,1,2335,58,0.975763,mixed,1.000000,1.000000,0.999144,1.000000,0.999572,56,2,0,2335,0.976927,0.976598,True
1,0890dcc046c079acc4de4202,aa94b19f-b5c4-474e-ba62-cf034f43347f,414,1,0,414,0.000000,all_negative,NaN,NaN,0.000000,0.000000,0.000000,414,0,0,0,0.008064,0.000000,False
4,7232bc533c21ce408d45d473,93aafd38-c91e-42a7-b67f-90287b9dd6f7,116,1,116,0,1.000000,all_positive,NaN,NaN,1.000000,0.500000,0.666667,0,0,58,58,0.563024,0.500000,False


Run stability:


,value
informative_runs,7
strongest_run_id,03b262cc-46f0-4837-a184-1eea22fdfdd0
strongest_run_pr_auc,1.0
weakest_informative_run_id,795ee584-86ed-4fe5-b317-3d022b850274
weakest_informative_run_machine,0890dcc046c079acc4de4202
minimum_informative_run_pr_auc,0.05699
minimum_informative_run_recall,0.0
mean_informative_run_pr_auc,0.633479
std_informative_run_pr_auc,0.374168


### 7. Compare train, validation, and final test without adapting the model

**What this cell does:** Combines existing development metrics with the new frozen test metrics and relates any change to notebook 17’s shift diagnosis.  
**Why it matters:** This gives context for generalization while preserving test as an assessment rather than a tuning source.  
**What to understand:** Missing train classification metrics remain missing instead of being recomputed or used for post-test decisions.

In [7]:
base=baseline_report.iloc[0]
redesign_models=pd.read_csv(REPORT_DIR/"validation_split_model_comparison.csv")
train_pr=float(redesign_models.query("candidate=='current_reference' and model_variant=='baseline'").iloc[0].train_pr_auc)
comparison=pd.DataFrame([
 {"split":"train","rows":int(split_summary.loc[split_summary.split.eq("train"),"total_rows"].iloc[0]),"positive_rate":float(split_summary.loc[split_summary.split.eq("train"),"positive_rate"].iloc[0]/100),"pr_auc":train_pr,"roc_auc":np.nan,"precision":np.nan,"recall":np.nan,"f1":np.nan},
 {"split":"validation","rows":int(split_summary.loc[split_summary.split.eq("validation"),"total_rows"].iloc[0]),"positive_rate":float(split_summary.loc[split_summary.split.eq("validation"),"positive_rate"].iloc[0]/100),"pr_auc":base.pr_auc,"roc_auc":base.roc_auc,"precision":base.precision,"recall":base.recall,"f1":base.f1},
 {"split":"test","rows":len(y_test),"positive_rate":global_metrics["positive_rate"],"pr_auc":global_metrics["pr_auc"],"roc_auc":global_metrics["roc_auc"],"precision":global_metrics["precision"],"recall":global_metrics["recall"],"f1":global_metrics["f1"]},
])
comparison["pr_auc_change_vs_validation"]=comparison.pr_auc-base.pr_auc
comparison["interpretation"]=np.select([comparison.split.eq("train"),comparison.pr_auc_change_vs_validation.abs().le(.02),comparison.pr_auc_change_vs_validation.gt(.02)],
 ["development_fit_reference","close_to_validation","improves_vs_validation"],default="degrades_vs_validation")
comparison.to_csv(REPORT_DIR/"final_train_validation_test_comparison.csv",index=False)
shift_class=shift_summary.loc[shift_summary.metric.eq("problem_classification"),"value"].iloc[0]
summary_rows=[
 ("model","final_model","Existing frozen baseline LightGBM","No retraining or post-test model selection"),
 ("threshold","decision_threshold",THRESHOLD,"Fixed prototype/development reference; not production optimized"),
 ("test","pr_auc",global_metrics["pr_auc"],"Primary ranking metric"),("test","roc_auc",global_metrics["roc_auc"],"Final untouched-test ROC-AUC"),
 ("test","precision",global_metrics["precision"],"At threshold 0.50"),("test","recall",global_metrics["recall"],"At threshold 0.50"),("test","f1",global_metrics["f1"],"At threshold 0.50"),
 ("test","false_negatives",int(fn),"Real next-five-minute slowdowns missed"),("test","false_positives",int(fp),"False alerts generated"),
 ("robustness","weakest_informative_run",weakest.run_id,f"PR-AUC={weakest.pr_auc:.6f}; machine={weakest.machine_id}"),
 ("robustness","run_pr_auc_std",run_stability["std_informative_run_pr_auc"],"Variation across informative mixed-class test runs"),
 ("limitation","distribution_shift",shift_class,"Known strong covariate shift and possible concept drift"),
 ("status","v1_status","experimental_prototype","Suitable for project prototype/demo, not production-grade reliability"),
 ("future","required_improvement","more_representative_diverse_operating_regime_data","Additional tuning alone is not sufficient"),
 ("guardrail","post_test_changes","none","No model, preprocessing, feature, split, label, calibration, or threshold changes"),
]
final_v1_summary=pd.DataFrame(summary_rows,columns=["section","metric","value","interpretation"])
final_v1_summary.to_csv(REPORT_DIR/"final_v1_summary.csv",index=False)
display(comparison); display(final_v1_summary)

,split,rows,positive_rate,pr_auc,roc_auc,precision,recall,f1,pr_auc_change_vs_validation,interpretation
0,train,82860,0.340309,0.999996,NaN,NaN,NaN,NaN,0.031008,development_fit_reference
1,validation,22212,0.491311,0.968988,0.958868,0.895394,0.899661,0.897523,0.000000,close_to_validation
2,test,27944,0.406384,0.951936,0.954877,0.815435,0.922068,0.865479,-0.017053,close_to_validation


,section,metric,value,interpretation
0,model,final_model,Existing frozen baseline LightGBM,No retraining or post-test model selection
1,threshold,decision_threshold,0.5,Fixed prototype/development reference; not pro...
2,test,pr_auc,0.951936,Primary ranking metric
3,test,roc_auc,0.954877,Final untouched-test ROC-AUC
4,test,precision,0.815435,At threshold 0.50
5,test,recall,0.922068,At threshold 0.50
6,test,f1,0.865479,At threshold 0.50
7,test,false_negatives,885,Real next-five-minute slowdowns missed
8,test,false_positives,2370,False alerts generated
9,robustness,weakest_informative_run,795ee584-86ed-4fe5-b317-3d022b850274,PR-AUC=0.056990; machine=0890dcc046c079acc4de4202


### 8. Create final evaluation figures

**What this cell does:** Saves the confusion matrix, PR/ROC curves, and machine/run performance figures.  
**Why it matters:** The plots make ranking quality, classification errors, and regime variability visible.  
**What to understand:** Curves are evaluation-only; no point on either curve is used to choose a new threshold.

In [8]:
plt.style.use("seaborn-v0_8-whitegrid")
cm=np.array([[tn,fp],[fn,tp]]); fig,ax=plt.subplots(figsize=(6,5)); image=ax.imshow(cm,cmap="Blues")
for i in range(2):
 for j in range(2): ax.text(j,i,f"{cm[i,j]:,}",ha="center",va="center",fontsize=16,color="white" if cm[i,j]>cm.max()/2 else "black")
ax.set(xticks=[0,1],yticks=[0,1],xticklabels=["Predicted no slowdown","Predicted slowdown"],yticklabels=["Actual no slowdown","Actual slowdown"],xlabel="Prediction at 0.50",ylabel="True label",title="Final untouched-test confusion matrix")
fig.colorbar(image,ax=ax); fig.tight_layout(); fig.savefig(FIGURE_DIR/"confusion_matrix.png",dpi=170); plt.close(fig)

pr_precision,pr_recall,_=precision_recall_curve(y_test,raw_model_score); fig,ax=plt.subplots(figsize=(7,5)); ax.plot(pr_recall,pr_precision,label=f"PR-AUC={global_metrics['pr_auc']:.4f}"); ax.axhline(y_test.mean(),ls="--",color="gray",label=f"prevalence={y_test.mean():.4f}"); ax.set(xlabel="Recall",ylabel="Precision",title="Final test precision–recall curve"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURE_DIR/"precision_recall_curve.png",dpi=170); plt.close(fig)
fpr_curve,tpr_curve,_=roc_curve(y_test,raw_model_score); fig,ax=plt.subplots(figsize=(7,5)); ax.plot(fpr_curve,tpr_curve,label=f"ROC-AUC={global_metrics['roc_auc']:.4f}"); ax.plot([0,1],[0,1],ls="--",color="gray"); ax.set(xlabel="False-positive rate",ylabel="True-positive rate",title="Final test ROC curve"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURE_DIR/"roc_curve.png",dpi=170); plt.close(fig)

fig,ax=plt.subplots(figsize=(10,5)); x=np.arange(len(by_machine)); w=.25
for offset,col,color in [(-w,"pr_auc","#4c78a8"),(0,"recall","#f58518"),(w,"f1","#54a24b")]: ax.bar(x+offset,by_machine[col],width=w,label=col,color=color)
ax.set_xticks(x,by_machine.machine_id,rotation=20,ha="right"); ax.set_ylim(0,1.05); ax.set_title("Final test performance by machine"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURE_DIR/"performance_by_machine.png",dpi=170); plt.close(fig)

ordered=by_run.sort_values("pr_auc",na_position="last"); fig,ax=plt.subplots(figsize=(11,6)); colors=["#e45756" if rid==weakest.run_id else "#4c78a8" for rid in ordered.run_id]; ax.barh([r[:8]+"…" for r in ordered.run_id],ordered.pr_auc,color=colors); ax.set_xlim(0,1.05); ax.set_xlabel("PR-AUC (missing for single-class runs)"); ax.set_title("Final test PR-AUC by run"); fig.tight_layout(); fig.savefig(FIGURE_DIR/"performance_by_run.png",dpi=170); plt.close(fig)
print(f"Saved {len(list(FIGURE_DIR.glob('*.png')))} final-test figures.")

Saved 5 final-test figures.


### 9. Save final V1 metadata and prove no frozen artifact changed

**What this cell does:** Writes a JSON manifest referencing—not copying or retraining—the frozen artifacts, embeds final metrics and limitations, and repeats all hashes.  
**Why it matters:** Dashboard integration needs a precise, auditable contract for model score, feature pipeline, threshold status, and known limitations.  
**What to understand:** Passing assertions confirm that evaluation produced reports only and did not mutate any frozen input after observing test results.

In [9]:
metadata={
 "model_name":"LightGBM","model_role":"frozen_baseline_final_v1_prototype","task":"slowdown_in_5min prediction",
 "created_at_utc":datetime.now(timezone.utc).isoformat(),"dataset":{"cleaned_measurements":cleaned_rows,"valid_modeling_rows":valid_modeling_rows,"machines":int(pd.read_csv(PATHS["feature_dataset"],usecols=["machine_id"]).machine_id.nunique()),"modeling_runs":int(split_runs.run_id.nunique()),"candidate_features":candidate_count,"retained_original_features":retained_count,"transformed_features":X_test.shape[1]},
 "artifacts":{"model_path":str(PATHS["model"].relative_to(PROJECT_ROOT)),"tree_preprocessor_path":str(PATHS["preprocessor"].relative_to(PROJECT_ROOT)),"feature_metadata_path":str(PATHS["feature_metadata"].relative_to(PROJECT_ROOT)),"x_test_path":str(PATHS["x_test"].relative_to(PROJECT_ROOT)),"model_sha256":frozen_before["model"],"preprocessor_sha256":frozen_before["preprocessor"]},
 "model_parameters":model.get_params(),"decision_threshold":THRESHOLD,"threshold_status":"prototype/development reference; not optimized or validated for production",
 "score_contract":{"raw_model_score":"predict_proba(X)[:, 1]","risk_score":"100 * raw_model_score","display_wording":"Model risk score for slowdown in the next 5 minutes","calibration_warning":"Do not describe as a perfectly calibrated probability."},
 "rule_c_reference":{"notebooks":["notebooks/06_label_definition.ipynb","notebooks/07_future_label_creation.ipynb"],"description":"Provisional Rule C: one sustained severe resource signal or at least two sustained moderate resource-group signals; future target searches strictly within segment over five minutes."},
 "final_test_metrics":{k:(v.item() if isinstance(v,np.generic) else v) for k,v in global_metrics.items()},"run_stability":{k:(v.item() if isinstance(v,np.generic) else v) for k,v in run_stability.items()},
 "known_limitations":["Experimental/prototype system.","Performance varies between machine/run operating regimes.","Distribution-shift analysis found strong covariate shift and possible concept drift.","Current data supports a project prototype/demo but not production-grade reliability across arbitrary machines.","Future improvement requires representative data from diverse operating regimes rather than only more hyperparameter tuning.","Threshold 0.50 is a fixed demo reference and is not production optimized.","Risk score is not guaranteed to be a calibrated probability."],
 "post_test_policy":"No model, hyperparameter, preprocessing, feature engineering, split, Rule C, threshold, or calibration changes may be made based on this test evaluation."
}
with open(MODEL_DIR/"model_metadata.json","w",encoding="utf-8") as handle: json.dump(metadata,handle,indent=2)

frozen_after={name:sha256(PATHS[name]) for name in frozen_names}; assert frozen_before==frozen_after,"Frozen artifact changed during final evaluation."
required=["final_test_metrics.csv","final_test_predictions.csv","final_test_by_machine.csv","final_test_by_run.csv","final_train_validation_test_comparison.csv","final_v1_summary.csv"]
for name in required:
 p=REPORT_DIR/name; assert p.exists() and p.stat().st_size>0 and len(pd.read_csv(p))>0
assert len(list(FIGURE_DIR.glob("*.png")))==5 and (MODEL_DIR/"model_metadata.json").exists()
print("FINAL V1 TEST EVALUATION COMPLETE")
print(f"PR-AUC={global_metrics['pr_auc']:.6f}; ROC-AUC={global_metrics['roc_auc']:.6f}; precision={global_metrics['precision']:.6f}; recall={global_metrics['recall']:.6f}; F1={global_metrics['f1']:.6f}")
print(f"TN={tn:,}; FP={fp:,}; FN={fn:,}; TP={tp:,}")
print(f"Weakest informative run: {weakest.run_id}; PR-AUC={weakest.pr_auc:.6f}")
print("Frozen model/preprocessing/data/split unchanged: True")
print("STOP: no tuning, calibration, SHAP, retraining, threshold change, or dashboard implementation was performed.")

FINAL V1 TEST EVALUATION COMPLETE
PR-AUC=0.951936; ROC-AUC=0.954877; precision=0.815435; recall=0.922068; F1=0.865479
TN=14,218; FP=2,370; FN=885; TP=10,471
Weakest informative run: 795ee584-86ed-4fe5-b317-3d022b850274; PR-AUC=0.056990
Frozen model/preprocessing/data/split unchanged: True
STOP: no tuning, calibration, SHAP, retraining, threshold change, or dashboard implementation was performed.
